# TV2-DE-07: Canonical Exploratory Data Analysis (EDA)

**Task ID:** TV2-DE-07 — Exploratory Data Analysis and Data Engineering Handoff  
**Producer:** Member 2 (TV2) — Data Engineering & Data Pipeline  
**Consumers:** Member 1 (TV1 — Modeling), Member 3 (TV3 — Dashboard)  

---

## Overview and Scope
This notebook provides a reproducible, lightweight narrative entry point into the canonical Exploratory Data Analysis for the Home Credit Default Risk dataset. All underlying analytical computations and plotting routines are imported directly from `src.data.eda` to maintain strict DRY (Don't Repeat Yourself) consistency across pipeline modules, reports, and notebooks.

> **Data Contract Note:** This analysis operates exclusively on the canonical dataset (`data/processed/cleaned_dataset.parquet`). Raw `application_train.csv` is accessed solely for the `DAYS_EMPLOYED` before/after cleaning audit. `application_test.csv` is strictly excluded to prevent data leakage.

> **Downstream Dependency Note:** Formal Fairness Checks (disparate impact, FPR parity) and Decision Threshold Confusion Matrices belong to task **TV2-DE-08** and are **BLOCKED / PENDING** until Member 1 (TV1) completes model training and provides scored prediction probabilities.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Ensure project root is in sys.path
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data.eda import (
    get_default_paths,
    compute_target_distribution,
    compute_income_summary,
    compute_age_group_summary,
    compute_categorical_default_rate,
    compute_spearman_correlation,
    compute_days_employed_audit,
    validate_eda_input_dataframe,
)

paths = get_default_paths()
print("Canonical dataset path:", paths["dataset_path"])
df = pd.read_parquet(paths["dataset_path"])
validate_eda_input_dataframe(df)
print(f"Loaded canonical dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")

---
## 1. Portfolio Target Distribution & Class Imbalance
The portfolio default label (`TARGET`) is binary with ~8.07% observed default rate.

In [ ]:
target_dist = compute_target_distribution(df)
pd.DataFrame([{
    "Non-Default (0)": f"{target_dist['count_0']:,} ({target_dist['rate_0']*100:.2f}%)",
    "Default (1)": f"{target_dist['count_1']:,} ({target_dist['rate_1']*100:.2f}%)",
    "Total Portfolio": f"{target_dist['total_count']:,}",
    "Default Rate": f"{target_dist['default_rate_pct']:.4f}%",
}])

---
## 2. Figure 01: Income Distribution by Target
Applicants who defaulted exhibit a slightly lower median income (135,000 CZK vs 148,500 CZK), though distributions exhibit substantial overlap.

![Figure 01: Income Distribution by Target](../reports/figures/eda/01_income_distribution_by_target.png)

In [ ]:
income_summary = compute_income_summary(df)
pd.DataFrame([
    {
        "Group": "Target 0 (Non-Default)",
        "Sample Size (N)": f"{income_summary['target_0']['count']:,}",
        "Median (CZK)": f"{income_summary['target_0']['median']:,.0f}",
        "IQR (CZK)": f"{income_summary['target_0']['iqr']:,.0f}",
        "Q25 (CZK)": f"{income_summary['target_0']['q25']:,.0f}",
        "Q75 (CZK)": f"{income_summary['target_0']['q75']:,.0f}",
    },
    {
        "Group": "Target 1 (Default)",
        "Sample Size (N)": f"{income_summary['target_1']['count']:,}",
        "Median (CZK)": f"{income_summary['target_1']['median']:,.0f}",
        "IQR (CZK)": f"{income_summary['target_1']['iqr']:,.0f}",
        "Q25 (CZK)": f"{income_summary['target_1']['q25']:,.0f}",
        "Q75 (CZK)": f"{income_summary['target_1']['q75']:,.0f}",
    }
])

---
## 3. Figure 02: Default Rate by Age Group
Observed default rate decreases across the chosen age brackets using the canonical persisted derived feature `AGE_GROUP` (constructed from `AGE_YEARS` with authoritative boundaries `[0, 25, 35, 45, 55, 65, 120]`, `right=False`), decreasing from 12.29% for Under 25 to 3.66% for 65+.

![Figure 02: Default Rate by Age Group](../reports/figures/eda/02_default_rate_by_age_group.png)


In [ ]:
age_agg = compute_age_group_summary(df)
age_agg[["age_group", "customer_count", "default_count", "non_default_count", "default_rate_pct"]].rename(columns={
    "age_group": "Age Group",
    "customer_count": "Total Applicants",
    "default_count": "Defaults",
    "non_default_count": "Non-Defaults",
    "default_rate_pct": "Default Rate (%)",
})


---
## 4. Figure 03: Default Rate by Occupation and Contract Type
Default rates vary substantially across occupation categories (Low-skill Laborers at 17.15% vs Accountants at 4.83%). Missing/Unknown occupation (31.35%) carries a 6.51% default rate and is preserved without demographic inference. Cash loans carry an 8.35% default rate vs 5.48% for Revolving loans.

![Figure 03: Default Rate by Occupation and Contract](../reports/figures/eda/03_default_rate_by_occupation_and_contract.png)


In [ ]:
occ_agg = compute_categorical_default_rate(df, "OCCUPATION_TYPE")
print("Top 5 highest default rate occupations:")
print(occ_agg.head(5)[["category", "total_count", "default_rate_pct"]].to_string(index=False))
print("\nContract type summary:")
contract_agg = compute_categorical_default_rate(df, "NAME_CONTRACT_TYPE")
print(contract_agg[["category", "total_count", "default_rate_pct"]].to_string(index=False))

---
## 5. Figure 04: Spearman Rank Correlation Heatmap
Key financial scale variables exhibit strong monotonic rank associations (AMT_CREDIT <-> AMT_GOODS_PRICE: 0.98, AMT_CREDIT <-> AMT_ANNUITY: 0.83). Note: TARGET is omitted to avoid supervised feature selection bias. High rank association is a descriptive modeling consideration for TV1, not proof of linear equivalence or automatic redundancy.

![Figure 04: Spearman Heatmap](../reports/figures/eda/04_key_numeric_spearman_heatmap.png)


In [ ]:
corr = compute_spearman_correlation(df)
print("Strongest Spearman rank associations (|rho| >= 0.70):")
cols = list(corr.columns)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        c1, c2 = cols[i], cols[j]
        val = corr.loc[c1, c2]
        if abs(val) >= 0.70:
            print(f"  {c1} <-> {c2}: rho = {val:.4f}")


---
## 6. Figure 05: DAYS_EMPLOYED Sentinel Cleaning Audit
In raw data, 55,374 rows (~18.01%) contained the extreme anomaly value 365,243 days (~1,000 years). In the canonical dataset, all 55,374 values are cleanly replaced with NaN, and the anomaly is preserved in the binary indicator `DAYS_EMPLOYED_ANOM`. Non-sentinel values retain their canonical signed days (<=0); conversion to years in plotting is display-only.

![Figure 05: DAYS_EMPLOYED Before/After](../reports/figures/eda/05_days_employed_before_after.png)


In [ ]:
raw_df = pd.read_csv(paths["raw_train_path"], usecols=["SK_ID_CURR", "DAYS_EMPLOYED"])
audit = compute_days_employed_audit(raw_df, df)
pd.DataFrame([{
    "Raw Total Rows": f"{audit['raw_total_rows']:,}",
    "Raw Sentinel 365243": f"{audit['raw_sentinel_count']:,} ({audit['raw_sentinel_rate']*100:.2f}%)",
    "Clean Sentinel 365243": f"{audit['clean_sentinel_count']}",
    "Clean DAYS_EMPLOYED NaNs": f"{audit['clean_nan_count']:,}",
    "Clean DAYS_EMPLOYED_ANOM=1": f"{audit['anom_flag_count']:,}",
    "Cleaning Gate Valid": audit['is_cleaning_valid'],
}])

---
## 7. Data Engineering Handoff Status
- **TV1 (Modeling):** Ready for modeling. Use `cleaned_dataset.parquet`. Fit all scalers/encoders/imputers on train fold only. See `docs/data/tv2_data_handoff.md` for full contract.
- **TV3 (Dashboard):** Ready for descriptive dashboarding. Use `cleaned_dataset.parquet` and `data_dictionary.csv`. Predictive score distributions remain pending TV1 model outputs.
- **TV2-DE-08 Notice:** Fairness analysis and threshold evaluation will be executed once TV1 provides scored model predictions.